# Code generation & debugging assistant

Lesson 5.2, following the course's exercise. An LLM writes `process_data`, the tests check it, and the failures go back to the model until every test passes.

The task description only mentions `sum` and `average`. The expanded tests also expect `median`, `None` for empty lists, ignored non-numbers and a `ValueError` for unknown modes, so the model learns those requirements **from the test feedback** (agentic test-driven development).

**Run in Colab, setup cell first.** Uncomment a `MODEL = ...` line there to choose a model.

In [ ]:
# Setup: Colab's built-in models, no API key needed. Run this cell first.
from google.colab import ai
print("Available models:", ", ".join(ai.list_models()))

MODEL = None                              # Colab's default; uncomment a line below to choose
# MODEL = "google/gemini-2.5-flash"
# MODEL = "google/gemini-2.5-pro"
# MODEL = "google/gemini-3.8-flash"

def get_completion(messages):
    """Send chat messages to the model. Colab's generate_text takes one prompt, so roles are inlined."""
    prompt = "\n\n".join(f"{m['role'].upper()}:\n{m['content']}" for m in messages) + "\n\nASSISTANT:\n"
    return ai.generate_text(prompt, model_name=MODEL) if MODEL else ai.generate_text(prompt)

## 1 · Task and test cases

In [1]:
task_description = """
Create a Python function called `process_data` that analyzes numerical data.
This function should accept a list of numbers and a mode ('sum' or 'average').
- In 'sum' mode, it should return the sum of the numbers.
- In 'average' mode, it should return the average.
"""

test_cases = [
    {"inputs": ([1, 2, 3, 4, 5], "sum"), "expected": 15},
    {"inputs": ([1, 2, 3, 4, 5], "average"), "expected": 3.0},
    {"inputs": ([11, 12, 13, 14, 15], "sum"), "expected": 65},
    {"inputs": ([11, 12, 13, 14, 15], "average"), "expected": 13.0},
    {"inputs": ([], "sum"), "expected": None},
    {"inputs": ([1, 3, 4], "median"), "expected": 3},
    {"inputs": ([1, 2, 3, 5], "median"), "expected": 2.5},
    {"inputs": ([1, 2, "a", 3], "sum"), "expected": 6},
    {"inputs": ([1, 2, None, 3, "b", 4], "average"), "expected": 2.5},
    {"inputs": ([10], "median"), "expected": 10},
    {"inputs": ([], "median"), "expected": None},
    {"inputs": ([1, 2, 3, 4, 5], "invalid_mode"), "expected": ValueError},
]

## 2 · Run the code against the tests

In [2]:
def execute_code(code, test_cases):
    """Run the generated code and check it against every test case."""
    results = {"execution_error": None, "test_results": [], "passed": 0, "failed": 0}
    namespace = {}
    try:
        exec(code, namespace)
    except Exception as e:
        results["execution_error"] = str(e)
        results["failed"] = len(test_cases)
        return results

    for i, test in enumerate(test_cases, start=1):
        expected = test["expected"]
        try:
            actual = namespace["process_data"](*test["inputs"])
            passed = actual == expected
        except Exception as e:
            actual = f"{type(e).__name__}: {e}"
            passed = isinstance(expected, type) and isinstance(e, expected)   # the expected exception
        results["passed" if passed else "failed"] += 1
        results["test_results"].append({"test_id": i, "inputs": test["inputs"], "expected": expected,
                                        "actual": actual, "passed": passed})
    return results


def format_feedback(results):
    """Turn the results into feedback text for the model."""
    if results["execution_error"]:
        return f"The code failed to run: {results['execution_error']}"
    feedback = f"Test Results: {results['passed']} passed, {results['failed']} failed\n"
    for test in results["test_results"]:
        if not test["passed"]:
            feedback += (f"\nTest #{test['test_id']}:\n  Inputs: {test['inputs']}\n"
                         f"  Expected: {test['expected']}\n  Actual: {test['actual']}\n")
    return feedback

A first version that only does what the task description asks, against the expanded tests:

In [3]:
first_version = """
def process_data(numbers, mode='average'):
    if mode == 'sum':
        return sum(numbers)
    return sum(numbers) / len(numbers)
"""
print(format_feedback(execute_code(first_version, test_cases)))

Test Results: 5 passed, 7 failed

Test #5:
  Inputs: ([], 'sum')
  Expected: None
  Actual: 0

Test #6:
  Inputs: ([1, 3, 4], 'median')
  Expected: 3
  Actual: 2.6666666666666665

Test #7:
  Inputs: ([1, 2, 3, 5], 'median')
  Expected: 2.5
  Actual: 2.75

Test #8:
  Inputs: ([1, 2, 'a', 3], 'sum')
  Expected: 6
  Actual: TypeError: unsupported operand type(s) for +: 'int' and 'str'

Test #9:
  Inputs: ([1, 2, None, 3, 'b', 4], 'average')
  Expected: 2.5
  Actual: TypeError: unsupported operand type(s) for +: 'int' and 'NoneType'

Test #11:
  Inputs: ([], 'median')
  Expected: None
  Actual: ZeroDivisionError: division by zero

Test #12:
  Inputs: ([1, 2, 3, 4, 5], 'invalid_mode')
  Expected: <class 'ValueError'>
  Actual: 3.0



## 3 · The feedback loop

In [ ]:
import re

def extract_code(response):
    # change: the course looks for a line that is exactly "```python", which fails on
    # "```py" or a trailing space; this takes any fenced block, or the whole reply
    match = re.search(r"```(?:python|py)?\s*\n(.*?)```", response, re.S)
    return match.group(1) if match else response


initial_prompt = f"""
You are an expert Python developer. Please write a Python function based on the following requirements:
{task_description}
Write only the function surrounded by ```python and ``` without any additional explanations.
"""

iterations = []
current_code = extract_code(get_completion([{"role": "user", "content": initial_prompt}]))
current_results = execute_code(current_code, test_cases)

In [ ]:
max_retries = 3

for i in range(max_retries + 1):
    iterations.append({"iteration": i, "code": current_code,
                       "passed": current_results["passed"], "failed": current_results["failed"]})
    print(f"Iteration {i}: {current_results['passed']} passed, {current_results['failed']} failed")
    if current_results["failed"] == 0:
        print("✅ All test cases passed!")
        break
    if i == max_retries:
        break

    feedback_prompt = f"""
You are an expert Python developer. You wrote a function based on these requirements:
{task_description}
Here is your current implementation:
```python
{current_code}
```
I've tested your code and here are the results:
{format_feedback(current_results)}
Please improve your code to fix any issues and make sure it passes all test cases.
Write only the improved function surrounded by ```python and ```.
"""
    current_code = extract_code(get_completion([{"role": "user", "content": feedback_prompt}]))
    current_results = execute_code(current_code, test_cases)

print("\nFinal code:\n" + iterations[-1]["code"])